#  Tarea 1: Extracción de datos web — Resultados del examen de admisión de la UNMSM

In [1]:
%pip install --upgrade pip
%pip install pandas selenium webdriver-manager

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


Paso 1: Instalacion de packages

In [ ]:

import pandas as pd
import time
import requests
import base64          #En el HTML de UNMSM el nombre del postulante esta codificado en base64.


# Herramientas de Selenium
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException
from webdriver_manager.chrome import ChromeDriverManager


# Herramientas recomendados por codex-openai para web scraping

from bs4 import BeautifulSoup      #Para leer y analizar el HTML 
from urllib.parse import urljoin   #para unir enlaces relativos con la URL base    
     


Paso 2 : Inicializacion de navegador automatizado. (Prueba)

In [7]:
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service)

BASE_URL = "https://admision.unmsm.edu.pe/Website20262/A/A.html"
driver.get(BASE_URL)
driver.maximize_window()

Paso 3: Desarrollo de las funciones para la automatización de la extracción de datos de los resultados del examen de admisión UNMSM 2026-2 para todas las carreras.



In [8]:


# Carga de la página y parseo con BeautifulSoup


def get_soup(url):
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    resp.encoding = resp.apparent_encoding
    return BeautifulSoup(resp.text, "html.parser")


# Extracción y filtrado de enlaces html por carrera

def parse_career_links(base_url):
    soup = get_soup(base_url)
    enlaces = []
    for a in soup.find_all("a", href=True):
        href = a["href"].strip()
        if href.lower().endswith("results.html"):
            texto = a.get_text(strip=True)
            if texto:
                enlaces.append((texto, urljoin(base_url, href)))
    return enlaces


# Lectura de datos ofuscados en Base64

def decode_obfuscated(element):
    if element is None:
        return ""
    span = element.find("span", class_="obfuscated")
    if span and span.get("data-auth"):
        try:
            return base64.b64decode(span["data-auth"]).decode("utf-8")
        except Exception:
            return ""
    return ""


# Obtención de resultados de admisión por carrera

def parse_results_page(page_url, carrera_nombre):
    soup = get_soup(page_url)
    tabla = soup.find("table")
    if tabla is None:
        return []

    resultados = []
    for fila in tabla.find_all("tr")[1:]:
        celdas = fila.find_all("td")
        if len(celdas) < 6:
            continue

        codigo = celdas[0].get_text(strip=True)
        nombre = decode_obfuscated(celdas[1])
        puntaje = celdas[3].get("data-score", "")
        merito = celdas[4].get("data-merit", "")
        observacion = celdas[5].get_text(strip=True)
        ingreso = "SÍ" if "ALCANZÓ VACANTE" in observacion.upper() else "NO"

        resultados.append({
            "Código": codigo,
            "Nombre": nombre,
            "Carrera": carrera_nombre,
            "Puntaje": puntaje,
            "Mérito E.P": merito,
            "Observación": observacion,
            "Ingresó": ingreso,
        })
    return resultados


# Iteración sobre carreras y consolidacion de los datos en excel

resultados_totales = []
for nombre, url_carrera in parse_career_links(BASE_URL):
    registros = parse_results_page(url_carrera, nombre)
    resultados_totales.extend(registros)
    time.sleep(0.2)

if resultados_totales:
    df = pd.DataFrame(resultados_totales)
    excel_file = "resultados_unmsm.xlsx"
    df.to_excel(excel_file, index=False, engine="openpyxl")
